# HugMergeUI — Phase A: MMLU/GSM8K vs conflict metrics (Colab T4)

Runs the **full** Phase A set (22 same-domain pairs from Phase 5 + 6 domain-divergent pairs from Phase B = 28 total) on a rented T4 GPU, merging each pair with mergekit (TIES) and scoring it on MMLU + GSM8K accuracy, then correlating against the conflict / conflict_weighted / drift_magnitude scores already computed locally (embedded below, no external files needed).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

Checkpoints after every pair to `phase_a_full_results.json` (Drive if mounted, else `/content`) so a disconnect doesn't lose progress — just re-run from the top, it'll skip pairs already in the checkpoint file.


In [ ]:
!pip install -q "transformers==4.46.3" "mergekit==0.1.4" lm-eval pyyaml


In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"


In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = "/content/drive/MyDrive/hugmergeui_phase_a"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("Using Google Drive for checkpoints:", RESULTS_DIR)
except Exception as e:
    RESULTS_DIR = "/content"
    print("Drive mount skipped/unavailable, using local /content (lost on disconnect unless you download manually):", e)


In [ ]:
"""Pair definitions + the conflict/conflict_weighted/drift_magnitude/perplexity
scores already computed locally (Phase 5 + Phase B) - embedded here so this
notebook is fully self-contained and doesn\'t need any uploaded files."""

FAMILIES = {
    "qwen": {
        "base": "Qwen/Qwen2.5-0.5B",
        "finetunes": {
            "dolphin": "dphn/Dolphin3.0-Qwen2.5-0.5B",
            "capybara": "wulli/Qwen2.5-0.5B-sft-capybara",
            "instruct": "Qwen/Qwen2.5-0.5B-Instruct",
            "jayhyeon": "JayHyeon/Qwen2.5-0.5B-SFT",
            "qgallouedec": "qgallouedec/Qwen2.5-0.5B-SFT",
        },
    },
    "smollm2": {
        "base": "HuggingFaceTB/SmolLM2-360M",
        "finetunes": {
            "instruct": "HuggingFaceTB/SmolLM2-360M-Instruct",
            "michaelj1": "Michaelj1/finetune-smolLM2-360M",
            "cot": "prithivMLmods/SmolLM2-CoT-360M",
            "rickified": "Masorian06/SmolLM2-360M-Rickified",
            "michaelj1_wikitext": "Michaelj1/INSTRUCT_smolLM2-360M-finetuned-wikitext2-raw-v1",
        },
    },
}

DOMAIN_MODELS = {
    "qwen": {
        "base": "Qwen/Qwen2.5-0.5B",
        "math": "pngwn/qwen2.5-0.5b-gsm8k-sft",
        "code": "qgallouedec/Qwen2.5-0.5B-codeforces-SFT",
        "chat": "Qwen/Qwen2.5-0.5B-Instruct",
    },
    "smollm2": {
        "base": "HuggingFaceTB/SmolLM2-360M",
        "math": "qiuyu8290/SmolLM2-360M-sft-math-gsm8k",
        "code": "CodeAtCMU/SmolLM2-360M_full_sft_code_data_120K",
        "chat": "HuggingFaceTB/SmolLM2-360M-Instruct",
    },
}
DOMAIN_PAIRS = [("math", "code"), ("math", "chat"), ("code", "chat")]

import itertools

PAIRS = []
for family_name, spec in FAMILIES.items():
    base_repo = spec["base"]
    PAIRS.append({"family": family_name, "pair": "base+base2", "base_repo": base_repo,
                  "model_a": base_repo, "model_b": base_repo})
    for (name_a, repo_a), (name_b, repo_b) in itertools.combinations(spec["finetunes"].items(), 2):
        PAIRS.append({"family": family_name, "pair": f"{name_a}+{name_b}", "base_repo": base_repo,
                      "model_a": repo_a, "model_b": repo_b})

for family_name, spec in DOMAIN_MODELS.items():
    base_repo = spec["base"]
    for name_a, name_b in DOMAIN_PAIRS:
        PAIRS.append({"family": family_name, "pair": f"{name_a}+{name_b}", "base_repo": base_repo,
                      "model_a": spec[name_a], "model_b": spec[name_b]})

print(f"{len(PAIRS)} pairs queued")


In [ ]:
EXISTING_METRICS = {
    ("qwen", "base+base2"): {'conflict': 0.0, 'conflict_weighted': 0.0, 'drift_magnitude': 0.0, 'perplexity': 10.709006275334913},
    ("qwen", "dolphin+capybara"): {'conflict': 0.4819034700095539, 'conflict_weighted': 0.4706013614885889, 'drift_magnitude': 0.10463435572275591, 'perplexity': 12.348541749341774},
    ("qwen", "dolphin+instruct"): {'conflict': 0.4680710360697468, 'conflict_weighted': 0.44383905908295884, 'drift_magnitude': 0.15637741271167183, 'perplexity': 13.037351150631768},
    ("qwen", "dolphin+jayhyeon"): {'conflict': 0.4917157965876414, 'conflict_weighted': 0.4874258659960633, 'drift_magnitude': 0.09446855805091871, 'perplexity': 11.937954590360748},
    ("qwen", "dolphin+qgallouedec"): {'conflict': 0.4972653851870217, 'conflict_weighted': 0.49578401151594104, 'drift_magnitude': 0.08405837724766935, 'perplexity': 11.725578830053808},
    ("qwen", "capybara+instruct"): {'conflict': 0.49482520070989533, 'conflict_weighted': 0.4917081428831225, 'drift_magnitude': 0.10298956422057405, 'perplexity': 11.054336083808465},
    ("qwen", "capybara+jayhyeon"): {'conflict': 0.4779832536761146, 'conflict_weighted': 0.465645256663858, 'drift_magnitude': 0.04108070955982093, 'perplexity': 11.34642451645453},
    ("qwen", "capybara+qgallouedec"): {'conflict': 0.4547551641846531, 'conflict_weighted': 0.43025127669999486, 'drift_magnitude': 0.030670528756571582, 'perplexity': 11.53352813551052},
    ("qwen", "instruct+jayhyeon"): {'conflict': 0.496015079263692, 'conflict_weighted': 0.49407006216437716, 'drift_magnitude': 0.09282376654873685, 'perplexity': 10.867297673991272},
    ("qwen", "instruct+qgallouedec"): {'conflict': 0.4982906403368072, 'conflict_weighted': 0.497367875240747, 'drift_magnitude': 0.08241358574548749, 'perplexity': 10.962627437191076},
    ("qwen", "jayhyeon+qgallouedec"): {'conflict': 0.46303501244912343, 'conflict_weighted': 0.44392375219978303, 'drift_magnitude': 0.020504731084734374, 'perplexity': 11.157292339284247},
    ("smollm2", "base+base2"): {'conflict': 0.0, 'conflict_weighted': 0.0, 'drift_magnitude': 0.0, 'perplexity': 9.562887296582769},
    ("smollm2", "instruct+michaelj1"): {'conflict': 0.4994003177630975, 'conflict_weighted': 0.4989740265701382, 'drift_magnitude': 0.08102884291718844, 'perplexity': 10.08134665602452},
    ("smollm2", "instruct+cot"): {'conflict': 0.49454744598166595, 'conflict_weighted': 0.49150758522124227, 'drift_magnitude': 0.08291005408238114, 'perplexity': 10.949204149649733},
    ("smollm2", "instruct+rickified"): {'conflict': 0.4932852118802593, 'conflict_weighted': 0.4896997489770168, 'drift_magnitude': 0.07874584156263759, 'perplexity': 9.887385391580377},
    ("smollm2", "instruct+michaelj1_wikitext"): {'conflict': 0.002638347348243534, 'conflict_weighted': 4.067227678162614e-05, 'drift_magnitude': 0.15234634961195292, 'perplexity': 9.8672890410236},
    ("smollm2", "michaelj1+cot"): {'conflict': 0.49997699331789064, 'conflict_weighted': 0.4999258466876733, 'drift_magnitude': 0.011590391245176086, 'perplexity': 14.985430201943585},
    ("smollm2", "michaelj1+rickified"): {'conflict': 0.4995912269885328, 'conflict_weighted': 0.49922908666351584, 'drift_magnitude': 0.007426178725432521, 'perplexity': 11.505674376893808},
    ("smollm2", "michaelj1+michaelj1_wikitext"): {'conflict': 0.4913785405029698, 'conflict_weighted': 0.48937359829751503, 'drift_magnitude': 0.08102668677474786, 'perplexity': 10.526074523407672},
    ("smollm2", "cot+rickified"): {'conflict': 0.4873507728746461, 'conflict_weighted': 0.4788696728793622, 'drift_magnitude': 0.009307389890625213, 'perplexity': 13.77589429783423},
    ("smollm2", "cot+michaelj1_wikitext"): {'conflict': 0.49475804994665346, 'conflict_weighted': 0.49161761096879164, 'drift_magnitude': 0.08290789793994055, 'perplexity': 11.040950546733514},
    ("smollm2", "rickified+michaelj1_wikitext"): {'conflict': 0.49353032321518125, 'conflict_weighted': 0.4898842954251891, 'drift_magnitude': 0.078743685420197, 'perplexity': 10.028362685807945},
    ("qwen", "math+code"): {'conflict': 0.4989208818636425, 'conflict_weighted': 0.49811461396320145, 'drift_magnitude': 0.01104157612155089, 'perplexity': 10.832357789680076},
    ("qwen", "math+chat"): {'conflict': 0.4991072695137651, 'conflict_weighted': 0.4972941108306711, 'drift_magnitude': 0.08475479104842083, 'perplexity': 11.510513242637415},
    ("qwen", "code+chat"): {'conflict': 0.49944054681730893, 'conflict_weighted': 0.4987107603738949, 'drift_magnitude': 0.08101940628262001, 'perplexity': 11.39642145223284},
    ("smollm2", "math+code"): {'conflict': 0.4994755727069683, 'conflict_weighted': 0.4987639626596408, 'drift_magnitude': 0.008267404341706823, 'perplexity': 9.596865721004141},
    ("smollm2", "math+chat"): {'conflict': 0.494121475897643, 'conflict_weighted': 0.49018536387177825, 'drift_magnitude': 0.07850342752891644, 'perplexity': 10.034424853217311},
    ("smollm2", "code+chat"): {'conflict': 0.48954333989832843, 'conflict_weighted': 0.4857174584057266, 'drift_magnitude': 0.08211248256718388, 'perplexity': 9.897241373639602},
}

print(f"{len(EXISTING_METRICS)} existing conflict/perplexity records loaded")


In [ ]:
"""MMLU/GSM8K limits. T4 has 16GB VRAM (vs 8GB locally) so these are pushed
higher than the local partial run for better resolution - tune down if a
pair is timing out."""

MMLU_NUM_FEWSHOT = 5
MMLU_LIMIT = 5       # questions per MMLU subject (57 subjects -> 285 total)
GSM8K_NUM_FEWSHOT = 8
GSM8K_LIMIT = 30


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import lm_eval
import yaml

CONFIGS_DIR = Path("/content/configs")
MERGED_DIR = Path("/content/merged")
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = Path(RESULTS_DIR) / "phase_a_full_results.json"


def build_config(base_repo: str, model_a: str, model_b: str) -> dict:
    return {
        "merge_method": "ties",
        "base_model": base_repo,
        "models": [
            {"model": model_a, "parameters": {"weight": 0.5, "density": 0.5}},
            {"model": model_b, "parameters": {"weight": 0.5, "density": 0.5}},
        ],
        "parameters": {"normalize": True},
        "dtype": "float32",
    }


def run_pair_eval(family: str, pair: str, base_repo: str, model_a: str, model_b: str) -> dict:
    pair_name = f"{family}__{pair}"
    config_path = CONFIGS_DIR / f"{pair_name}.yaml"
    out_path = MERGED_DIR / pair_name

    with open(config_path, "w") as f:
        yaml.safe_dump(build_config(base_repo, model_a, model_b), f)

    print(f"[{pair_name}] merging...", flush=True)
    subprocess.run(["mergekit-yaml", str(config_path), str(out_path), "--cuda"], check=True)

    model_args = f"pretrained={out_path.resolve()},dtype=float32"

    print(f"[{pair_name}] MMLU (limit={MMLU_LIMIT}/subject, {MMLU_NUM_FEWSHOT}-shot)...", flush=True)
    mmlu_res = lm_eval.simple_evaluate(
        model="hf", model_args=model_args, tasks=["mmlu"],
        num_fewshot=MMLU_NUM_FEWSHOT, limit=MMLU_LIMIT,
        batch_size="auto", device="cuda", log_samples=False,
    )
    mmlu_acc = mmlu_res["results"]["mmlu"]["acc,none"]

    print(f"[{pair_name}] GSM8K (limit={GSM8K_LIMIT}, {GSM8K_NUM_FEWSHOT}-shot)...", flush=True)
    gsm8k_res = lm_eval.simple_evaluate(
        model="hf", model_args=model_args, tasks=["gsm8k"],
        num_fewshot=GSM8K_NUM_FEWSHOT, limit=GSM8K_LIMIT,
        batch_size="auto", device="cuda", log_samples=False,
    )
    gsm8k_key = ("exact_match,strict-match" if "exact_match,strict-match" in gsm8k_res["results"]["gsm8k"]
                 else "exact_match,flexible-extract")
    gsm8k_acc = gsm8k_res["results"]["gsm8k"][gsm8k_key]

    shutil.rmtree(out_path, ignore_errors=True)

    base_metrics = EXISTING_METRICS.get((family, pair), {})
    result = {
        "family": family, "pair": pair, "model_a": model_a, "model_b": model_b,
        "mmlu_acc": mmlu_acc, "gsm8k_acc": gsm8k_acc,
        **base_metrics,
    }
    print(f"[{pair_name}] done: {result}", flush=True)
    return result


In [ ]:
results = []
if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    print(f"Resuming: {len(results)} pairs already done, skipping those.")

done_keys = {(r["family"], r["pair"]) for r in results}

for p in PAIRS:
    key = (p["family"], p["pair"])
    if key in done_keys:
        continue
    result = run_pair_eval(p["family"], p["pair"], p["base_repo"], p["model_a"], p["model_b"])
    results.append(result)
    RESULTS_PATH.write_text(json.dumps(results, indent=2))

print(f"\n=== ALL {len(results)} PAIRS DONE ===")


In [ ]:
from scipy import stats

print(f"=== Phase A correlations (n={len(results)}) ===\n")
for target in ["mmlu_acc", "gsm8k_acc"]:
    target_vals = [r[target] for r in results]
    print(f"--- vs {target} ---")
    for metric in ["conflict", "conflict_weighted", "drift_magnitude"]:
        vals = [r[metric] for r in results]
        pr, pp = stats.pearsonr(vals, target_vals)
        sr, sp = stats.spearmanr(vals, target_vals)
        print(f"  {metric}: pearson r={pr:.4f} (p={pp:.4g}) | spearman rho={sr:.4f} (p={sp:.4g})")
    print()


In [ ]:
from google.colab import files
files.download(str(RESULTS_PATH))
